In [36]:
# IMPORTS
import pandas as pd
from pathlib import Path

# Build paths
pv_path = Path("pv_capacity_factor_raw.csv")
onshore_wind_path = Path("onshore_wind_capacity_factor_raw.csv")
offshore_wind_path = Path("offshore_wind_capacity_factor_raw.csv")

# Read in CSVs as dataframes
df_pv_raw = pd.read_csv(pv_path, delimiter=";")
df_onshore_wind_raw = pd.read_csv(onshore_wind_path, delimiter=";")
df_offshore_wind_raw = pd.read_csv(offshore_wind_path, delimiter=";")

# Convert ISO8601 to datetime
df_pv_raw["utc_time"] = pd.to_datetime(df_pv_raw["utc_time"], utc=True, errors="raise")
df_onshore_wind_raw["utc_time"] = pd.to_datetime(df_onshore_wind_raw["utc_time"], utc=True, errors="raise")
df_offshore_wind_raw["utc_time"] = pd.to_datetime(df_offshore_wind_raw["utc_time"], utc=True, errors="raise")


In [37]:
# Keep only time and Denmark
df_pv = df_pv_raw[["utc_time", "DNK"]]
df_onshore_wind = df_onshore_wind_raw[["utc_time", "DNK"]]
df_offshore_wind = df_offshore_wind_raw[["utc_time", "DNK"]]

# Keep only 2007-2017 
start = "2007-01-01"
end = "2017-12-31"

df_pv = df_pv[(df_pv["utc_time"] >= start) & (df_pv["utc_time"] <= end)]
df_onshore_wind = df_onshore_wind[(df_onshore_wind["utc_time"] >= start) & (df_onshore_wind["utc_time"] <= end)]
df_offshore_wind = df_offshore_wind[(df_offshore_wind["utc_time"] >= start) & (df_offshore_wind["utc_time"] <= end)]

df_pv = df_pv.rename(columns={"DNK": "cf"})
df_onshore_wind = df_onshore_wind.rename(columns={"DNK": "cf"})
df_offshore_wind = df_offshore_wind.rename(columns={"DNK": "cf"})

def hourly_climatology(df):
    df = df[~((df.utc_time.dt.month == 2) & (df.utc_time.dt.day == 29))]
    df["month"] = df["utc_time"].dt.month
    df["day"] = df["utc_time"].dt.day
    df["hour"] = df["utc_time"].dt.hour
    
    result = (
        df.groupby(["month","day","hour"])["cf"]
        .mean()
        .reset_index()
    )
    result["timestamp"] = pd.to_datetime(dict(year=2017, month=result["month"], day=result["day"], hour=result["hour"]))

    result = result.sort_values("timestamp").reset_index(drop=True)

    # Create formatted column
    result["datetime"] = result["timestamp"].dt.strftime("%d/%m %H:%M")

    return result

pv_profile = hourly_climatology(df_pv)
onshore_profile = hourly_climatology(df_onshore_wind)
offshore_profile = hourly_climatology(df_offshore_wind)

In [38]:
# Function to format and export climatology profiles
def export_profile(df, filename):
    
    # Keep only time and cf
    out = df[["datetime", "cf"]].copy()
    out["time"] = out["datetime"]
    out=out[["time", "cf"]]
    
    # Set time as index
    out = out.set_index("time")
    
    out.to_csv(filename)
    
    return out

pv_out = export_profile(pv_profile, "averaged_pv_capacity_factor_denmark_hourly.csv")
onshore_out = export_profile(onshore_profile, "averaged_onshore_wind_capacity_factor_denmark_hourly.csv")
offshore_out = export_profile(offshore_profile, "averaged_offshore_wind_capacity_factor_denmark_hourly.csv")


In [41]:
# Electricity demand

denmark_demand_ds = pd.read_csv("electricity_demand.csv", delimiter=";")
denmark_demand_ds["utc_time"] = pd.to_datetime(denmark_demand_ds["utc_time"], utc=True, errors="raise")
denmark_demand_ds = denmark_demand_ds.rename(columns={"DNK": "demand"})
denmark_demand_ds = denmark_demand_ds[["utc_time", "demand"]]
denmark_demand_ds = denmark_demand_ds[~((denmark_demand_ds.utc_time.dt.month == 2) & (denmark_demand_ds.utc_time.dt.day == 29))]
denmark_demand_ds["month"] = denmark_demand_ds["utc_time"].dt.month
denmark_demand_ds["day"] = denmark_demand_ds["utc_time"].dt.day
denmark_demand_ds["hour"] = denmark_demand_ds["utc_time"].dt.hour
result = (
    denmark_demand_ds.groupby(["month","day","hour"])["demand"]
    .mean()
    .reset_index()
)
result["timestamp"] = pd.to_datetime(dict(year=2015, month=result["month"], day=result["day"], hour=result["hour"]))
result = result.sort_values("timestamp").reset_index(drop=True)
# Create formatted column
result["datetime"] = result["timestamp"].dt.strftime("%d/%m %H:%M")
result["time"] = result["datetime"]
result = result[["time", "demand"]]
# Set time as index
out = result.copy()
out = out.set_index("time")
out.to_csv("denmark_demand.csv")


In [31]:
# IMPORTS
import pandas as pd
hydropower_ds = pd.read_csv("jrc-efas-hydropower-ror.csv", delimiter=",")


In [32]:
nor_hydro = hydropower_ds[hydropower_ds["country_iso2"] == "NO"]
gb_hydro = hydropower_ds[hydropower_ds["country_iso2"] == "GB"]

# model swedish hydro based on finnish as Finland is hydrologically similar to Sweden (Nordic, snowmelt-driven)
no_hydro_copy = hydropower_ds[hydropower_ds["country_iso2"] == "NO"].copy()

scale_se_from_no = 0.5

se_hydro = no_hydro_copy.copy()
se_hydro["country_iso2"] = "SE"
se_hydro["run_of_river_mwh"] = se_hydro["run_of_river_mwh"] * scale_se_from_no

In [33]:
hydro_all = pd.concat([nor_hydro, gb_hydro, se_hydro])
hydro_all["time"] = pd.to_datetime(hydro_all["date"])
daily_hydro = hydro_all.pivot(
    index="time",
    columns="country_iso2",
    values="run_of_river_mwh"
)
daily_hydro = daily_hydro.rename(columns={"time": "utc_time"})

In [34]:
hydro_hourly = (daily_hydro.resample("h").ffill() / 24.0)

In [35]:
hydro_hourly.to_csv("hydro_hourly.csv",index=True,index_label="utc_time")